In [42]:
%reload_ext autoreload

import numpy as np
import pickle
import pandas as pd
from collections import defaultdict

from data_generation.network_generator import StochasticBlockModel
from data_generation.data_simulator import IndividualDataSimulator
from constants import NetworkSettings, DataGenerationSettings
from data_generation.data_generator_main import run_data_simulations
from methods.methods_est import LinearRegressionEstimator
from methods.visualization import ResultsPlotter

In [12]:
network_settings = NetworkSettings()
dgp_settings = DataGenerationSettings()

In [4]:
results = run_data_simulations(
    assignment_types=dgp_settings.assignment_types,
    treatment_effects=dgp_settings.treatment_effects,
    network_configs=network_settings.networks_types,
    neighbour_influences=[0.8999999999999999],
    n_nodes=network_settings.n_nodes,
    n_features=dgp_settings.n_features,
    error_mean=dgp_settings.error_mean,
    error_std=dgp_settings.error_std,
    beta_mean=dgp_settings.beta_mean,
    beta_std=dgp_settings.beta_std,
    share_treatment=dgp_settings.share_treatment,
    n_edges=network_settings.n_edges,
    n_sim=dgp_settings.n_sim,
    add_network_features=True,
    add_embeddings=True,
    add_count_treated=True,
    network_structure_type="random",
    dimensions=network_settings.vec_dimensions,
    walk_length=network_settings.vec_walk_length,
    num_walks=network_settings.vec_num_walks,
    p_random_walk=network_settings.p_random_walk,
    q_random_walk=network_settings.q_random_walk,
    vec_window=network_settings.vec_window,
    min_count=network_settings.min_count,
    batch_words=network_settings.batch_words,
    corrupt_network=True,
    removal_prob=network_settings.removal_prob,
    addition_prob=network_settings.addition_prob
)

2025-03-28 14:48:15,381 - INFO - Checking validity...
2025-03-28 14:48:15,388 - INFO - Starting data simulations...
Generating walks (CPU: 1): 100%|██████████| 50/50 [00:00<00:00, 73.92it/s]
2025-03-28 14:48:18,657 - INFO - collecting all words and their counts
2025-03-28 14:48:18,658 - INFO - PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2025-03-28 14:48:18,702 - INFO - PROGRESS: at sentence #10000, processed 300000 words, keeping 100 word types
2025-03-28 14:48:18,751 - INFO - collected 100 word types from a corpus of 600000 raw words and 20000 sentences
2025-03-28 14:48:18,756 - INFO - Creating a fresh vocabulary
2025-03-28 14:48:18,759 - INFO - Word2Vec lifecycle event {'msg': 'effective_min_count=1 retains 100 unique words (100.00% of original 100, drops 0)', 'datetime': '2025-03-28T14:48:18.759821', 'gensim': '4.3.3', 'python': '3.12.6 (main, Sep 16 2024, 19:20:48) [Clang 12.0.5 (clang-1205.0.22.11)]', 'platform': 'macOS-15.3.1-x86_64-i386-64bit', 'event': 'pr

### Analysis

In [24]:
network_configs = network_settings.networks_types
neighbour_influences = dgp_settings.n_influence_list
feature_adj_sets = network_settings.feature_adj_sets

In [43]:
lr_estimator = LinearRegressionEstimator()
results = defaultdict(lambda: defaultdict(dict))
all_results_df = pd.DataFrame()
network_type = "barabasi_albert_graph"

for influence in neighbour_influences:
    file_path = f"/Users/polinarevina/Desktop/thesis_new/src/simulation_results/sim_random_random_barabasi_albert_graph_influence{influence}_corrupt_network.pkl"
    
    with open(file_path, "rb") as f:
        data = pickle.load(f)
    
    results_dict = defaultdict(lambda: {"pvalue": [], "coef": []})

    for i_sim in range(dgp_settings.n_sim):
        individ_data_sim = data[i_sim]["individ_data"]
        for feature_name, features_ps in feature_adj_sets.items():
            individ_data_sim = data[i_sim]["individ_data"]
            for feature_set, features in feature_adj_sets.items():
                coef, pvalue = lr_estimator.calculate_results(
                    individ_data=individ_data_sim,
                    additional_feature_names=features
                )
                results_dict[feature_set]["pvalue"].append(pvalue)
                results_dict[feature_set]["coef"].append(coef)

        results[network_type][influence] = dict(results_dict)

    for feature_set, features in feature_adj_sets.items():
        results_plotter = ResultsPlotter(
            estimated_effect=results_dict[feature_set]["pvalue"],
            true_effect=dgp_settings.treatment_effect_mean,
            network_type=network_type,
            assignment_type=assignment_type,
            neighbour_influence=influence,
            treatment_mean=dgp_settings.treatment_effect_mean,
            additional_feature_names=features
        )
        results_plotter.calculate_results()
        results_df = results_plotter.create_results()
        results_plotter.create_plots()
        all_results_df = pd.concat([all_results_df, results_df], ignore_index=True)


In [44]:
all_results_df

,Network,Assignment,Influence,Treatment,Mean Estimated,Mean True,Perc Diff (%),Feature Names
0,barabasi_albert_graph,random,0.3,0.3,0.267,0.3,11.771,None
1,barabasi_albert_graph,random,0.3,0.3,0.476,0.3,45.430,"[degree_centrality, betweenness_centrality, co..."
2,barabasi_albert_graph,random,0.3,0.3,0.465,0.3,43.171,"[degree_centrality, betweenness_centrality]"
3,barabasi_albert_graph,random,0.3,0.3,0.277,0.3,8.150,[community]
4,barabasi_albert_graph,random,0.3,0.3,0.473,0.3,44.779,"[emb_0, emb_1, emb_2, emb_3, emb_4, emb_5, emb..."
5,barabasi_albert_graph,random,0.6,0.3,0.181,0.3,49.420,None
6,barabasi_albert_graph,random,0.6,0.3,0.448,0.3,39.618,"[degree_centrality, betweenness_centrality, co..."
7,barabasi_albert_graph,random,0.6,0.3,0.434,0.3,36.556,"[degree_centrality, betweenness_centrality]"
8,barabasi_albert_graph,random,0.6,0.3,0.183,0.3,48.613,[community]
9,barabasi_albert_graph,random,0.6,0.3,0.467,0.3,43.466,"[emb_0, emb_1, emb_2, emb_3, emb_4, emb_5, emb..."
